In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide dataset

This notebook curates the **BiToxNet** dataset from multiple CSV files. Peptide sequences and their toxicity labels are collected from different subdatasets, standardized into a unified schema, and subjected to quality control to produce a curated dataset suitable for downstream analysis.

- **Toxic effect / endpoint:** neurotoxic  
- **Source:** BiToxNet  
- **Sequence scope:** only peptide sequences are considered (protein sequences are excluded).

The pipeline performs the following steps:

- **Reads all CSV files** from the BiToxNet input directory and its subfolders.
- **Excludes protein-level datasets**, retaining only peptide-level data.
- **Standardizes column names**:
  - `Sequence` → `sequence`
  - `Label` → `label`
- **Concatenates all datasets** into a single dataframe.
- **Performs duplicate sequence quality control**:
  - unique sequences are retained,
  - duplicated sequences with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description file and appends QC statistics.
- **Exports curated outputs**:
  - `processed_neurotoxic_dataset.csv`
  - `metadata.json`

In [2]:
name_source = "BiToxNet"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
base_path = Path(PATH_INPUT) / name_source

for file in base_path.rglob("*.csv"):  
    if "Protein" in file.parts:         # excluir carpeta Protein
        continue
    df = pd.read_csv(file)
    df = df.rename(columns={"Sequence": "sequence", "Label": "label"})
    dfs.append(df)
df_bixtoxnet = pd.concat(dfs, ignore_index=True)

In [4]:
df_bixtoxnet

,sequence,label
0,DCLGWFSGCDPNNNKCCEGYVCHWKYPWCRYDL,1
1,LSKKQCGADGQFCFLPGLGLNCCSGLCLIVCVPT,1
2,RGCCNGRGGCSSRWCRDHARCC,1
3,GCIATGSVCTLSKGCCTKNCGWNFKCNPPNQ,1
4,GCISTGSFCTLSKGCCTKNCGWNFKCNPPNQ,1
...,...,...
9529,GPPCCLYGSCRPFPGCYNALCCRK,1
9530,VGIPVSCKHSGQCIKPCKDAGMRFGKCMNRKCDCTPK,1
9531,CCQWPCSHGCIPCCY,1
9532,GPSFCKADEKPCEYHSDCCNCCLSGICAPSTNWILPGCSTSSFFKI,1


- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_bixtoxnet, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(7780, 2)

In [7]:
df_errors.shape

(0, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_bixtoxnet)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2026,
 'last update date': datetime.datetime(2025, 11, 9, 0, 0),
 'download date': Timestamp('2026-02-26 00:00:00'),
 'file format': 'csv',
 'peptide property': 'neurotoxic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot, Previously published model dataset',
 'repository or server': 'https://github.com/Feng106-w/BiToxNet',
 'publication': 'https://link.springer.com/article/10.1186/s12915-026-02508-8',
 'number_of_raw_sequences': 9534,
 'number_of_sequences_retained': 7780,
 'number_of_positive_sequences': 2399,
 'number_of_negative_sequences': 5381,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_neurotoxic_dataset.csv", index=False)